# 07 — Final test: test seti, bir kez
⚠️ **Test seti bir kez ölçülür; bu notebook'tan sonra hiçbir seçim değişmez.**
Sevk kararı (H5 Görev 16) verilmeden çalıştırma.
**Çalışma zamanı → A100** (bootstrap 12 çekirdekte paralel). Secrets: `KAGGLE_API_TOKEN`,
`GITHUB_TOKEN`. Oturum koparsa yeniden **Tümünü çalıştır**: biten sonuçlar atlanır.

In [ ]:
BRANCH = 'h5'
SHIPPED_PIPELINE = ''   # Görev 16'daki karar, ör. 'e4_late_fusion'
SHIPPED_VARIANT = ''    # ör. 'onnx_int8'
assert SHIPPED_PIPELINE and SHIPPED_VARIANT, 'önce sevk kararı (H5 Görev 16)'
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!uv pip install --system -q -e '/content/baykus/python[train]'
import sys
sys.path.insert(0, '/content/baykus/python/src')
!python --version && nproc && (nvidia-smi -L || true)

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
%%time
# Test bölümünün 4 kanallı TIFF'leri (E3, E3c; ≈ 2-3 dk, 12 çekirdek) ve modeller
import json, os
from pathlib import Path
from baykus.train.checkpoints import find_runs
NPROC = os.cpu_count()
!baykus-data rgbt4 --out /content/dv_rgbt --splits test --workers {NPROC}

def train_run(exp):
    done = find_runs(f'{DRIVE}/runs', exp, finished=True)
    return json.loads((done[-1] / 'baykus_train.json').read_text(encoding='utf-8'))['run_id'] if done else None

EXPS = ['e1_rgb', 'e2_ir', 'e3_early_fusion', 'e3c_modality_dropout']
RUN = {e: train_run(e) for e in EXPS}
for e, r in RUN.items():
    print(e, '→', r)
assert all(RUN.values()), 'biten eğitim eksik'
DONE = lambda name: Path(f'{DRIVE}/results/runs/{name}_ir.json').exists()

In [ ]:
%%time
# 1) E1, E2, E3, E3c test (iki etiket seti, 1000 bootstrap; model başına ≈ birkaç dk).
for e in EXPS:
    if DONE(f'{RUN[e]}_test'):
        print('zaten var:', e)
        continue
    W = f'{DRIVE}/runs/{RUN[e]}/weights/best.pt'
    !cd /content/baykus/python && baykus-eval run --config ../configs/{e}.yaml --weights "{W}" --data-root /content/dv_rgbt --meta /content/meta --split test --project {DRIVE}/eval --results {DRIVE}/results --workers {NPROC}

In [ ]:
%%time
# 2) E4 ve E4b test: val'de seçilen parametrelerle (yeniden seçim yok).
R = f'{DRIVE}/eval/{RUN["e1_rgb"]}_test/predictions_coco.json.gz'
I = f'{DRIVE}/eval/{RUN["e2_ir"]}_test/predictions_coco.json.gz'
for EXP in ['e4_late_fusion', 'e4b_late_fusion_illum']:
    !cd /content/baykus/python && baykus-fusion apply --config ../configs/fusion/{EXP}.yaml --selection ../results/fusion/{EXP}_val_selection.json --rgb-preds "{R}" --ir-preds "{I}" --data-root /content/dv_rgbt --meta /content/meta --split test --dataset dronevehicle --project {DRIVE}/eval --results {DRIVE}/results --workers {NPROC}

In [ ]:
%%time
# 3) Sevk edilen varyant test'te (ORT CPU, bütün çekirdekler; iki etiket seti, 1000 bootstrap).
import yaml
E = yaml.safe_load(Path('/content/baykus/configs/edge/e9_edge.yaml').read_text(encoding='utf-8'))
P = E['pipelines'][SHIPPED_PIPELINE]
V, MODELS_DIR, PROJ = SHIPPED_VARIANT, f'{DRIVE}/edge/models', f'{DRIVE}/edge/eval'
EXT = '.engine' if V.startswith('trt_') else '.onnx'
names = [P['model']] if P['kind'] == 'single' else [P['rgb'], P['ir']]
for k in names:
    r = E['models'][k]['run_id']
    !cd /content/baykus/python && baykus-edge eval --config ../configs/edge/e9_edge.yaml --model {k} --variant {V} --file "{MODELS_DIR}/{r}_{V}{EXT}" --data-root /content/dv_rgbt --meta /content/meta --split test --label-sets ir rgb --n-boot 1000 --workers {NPROC} --threads {NPROC} --project {PROJ} --results {DRIVE}/results
if P['kind'] == 'late_fusion':
    R = f"{PROJ}/{E['models'][P['rgb']]['run_id']}_test_{V}/predictions_coco.json.gz"
    I = f"{PROJ}/{E['models'][P['ir']]['run_id']}_test_{V}/predictions_coco.json.gz"
    !cd /content/baykus/python && baykus-fusion apply --config ../configs/fusion/e4_late_fusion.yaml --selection ../{P['selection']} --rgb-preds "{R}" --ir-preds "{I}" --data-root /content/dv_rgbt --meta /content/meta --split test --dataset dronevehicle --variant {V} --project {PROJ} --results {DRIVE}/results --workers {NPROC}